# 🏠 Project 02: House Price Predictor

**What you'll learn:**
- What **supervised machine learning** is
- The complete ML workflow: data → train → evaluate → predict
- How **Linear Regression** works
- How to measure model performance

**Time to complete:** ~30 minutes

---

## 🧠 The Big Picture

Imagine you want to estimate the price of a house. You know:
- Its size (square footage)
- Number of bedrooms
- How old it is
- How far it is from the city

**The idea:** Show the model hundreds of past house sales where we know the price, and it learns the pattern. Then it can estimate prices for houses it has never seen!

This is called **supervised learning** — we supervise the model by telling it the right answers during training.

---

## Step 1 — Install & Import Libraries

In [ ]:
import sys
!{sys.executable} -m pip install scikit-learn pandas matplotlib numpy --quiet
print('✅ Libraries ready!')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

np.random.seed(42)  # Makes results reproducible
print('✅ Imports done!')

## Step 2 — Create the Dataset

We'll generate a realistic **synthetic** dataset (fake but realistic data). In real projects, you'd load a CSV from Kaggle or a database.

Our dataset will have **300 houses** with these features:

| Feature | Description |
|---------|-------------|
| `size_sqft` | Size of the house in square feet |
| `bedrooms` | Number of bedrooms |
| `age_years` | How old the house is |
| `distance_km` | Distance from city center |
| `price` | 🎯 What we want to predict! |

In [ ]:
n_samples = 300

# Generate random features
size_sqft   = np.random.randint(500, 4001, n_samples).astype(float)
bedrooms    = np.random.randint(1, 7, n_samples).astype(float)
age_years   = np.random.randint(0, 51, n_samples).astype(float)
distance_km = np.random.uniform(1, 30, n_samples)

# Price formula (what the "real world" looks like)
# Bigger house   → higher price (+$50 per sq ft)
# More bedrooms  → higher price (+$10k per bedroom)
# Older house    → lower price  (-$2k per year)
# Further away   → lower price  (-$5k per km)
noise = np.random.normal(0, 20_000, n_samples)  # random variation
price = (
    50 * size_sqft
    + 10_000 * bedrooms
    - 2_000 * age_years
    - 5_000 * distance_km
    + 80_000
    + noise
).clip(min=50_000)

df = pd.DataFrame({
    'size_sqft':   size_sqft,
    'bedrooms':    bedrooms,
    'age_years':   age_years,
    'distance_km': distance_km.round(1),
    'price':       price.round(-3),
})

print(f'Dataset: {df.shape[0]} houses, {df.shape[1]} columns')
print()
df.head(8)

## Step 3 — Explore the Data

Always explore your data before training! This is called **Exploratory Data Analysis (EDA)**.

In [ ]:
print('📊 Basic Statistics:')
df.describe().round(1)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Feature Distributions', fontsize=14, fontweight='bold')

features = ['size_sqft', 'bedrooms', 'age_years', 'price']
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for ax, feat, color in zip(axes.flat, features, colors):
    ax.hist(df[feat], bins=20, color=color, edgecolor='white', alpha=0.8)
    ax.set_title(feat.replace('_', ' ').title())
    ax.set_ylabel('Count')
    if feat == 'price':
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

plt.tight_layout()
plt.show()

In [ ]:
# See how each feature correlates with price
# Values close to +1 or -1 mean strong relationship
print('📈 Correlation with Price (house price):')
correlations = df.corr()['price'].drop('price').sort_values(ascending=False)
for feat, corr in correlations.items():
    bar = '█' * int(abs(corr) * 20)
    direction = '+' if corr > 0 else '-'
    print(f'  {feat:<15}: {direction}{bar:<22} {corr:+.3f}')
print()
print('Interpretation: size_sqft has the strongest positive correlation with price.')

## Step 4 — Prepare the Data

### Train/Test Split
We split our data into two groups:
- **Training set (80%)** — the model learns from this
- **Test set (20%)** — we use this to check how well the model works on *unseen* data

> 💡 Think of it like a student studying from a textbook (training), then taking an exam with new questions (testing).

### Feature Scaling
We scale the features so they're all on a similar range. This helps the algorithm converge better.

In [ ]:
FEATURES = ['size_sqft', 'bedrooms', 'age_years', 'distance_km']
TARGET = 'price'

X = df[FEATURES]
y = df[TARGET]

# Split 80% train / 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # ⚠️ use transform (not fit_transform) here!

print(f'Training set : {X_train.shape[0]} houses')
print(f'Test set     : {X_test.shape[0]} houses')
print()
print('Feature means (after scaling → should all be ~0):')
print(X_train_scaled.mean(axis=0).round(5))

## Step 5 — Train the Model

### How Does Linear Regression Work?

It finds the best-fit line (or hyperplane) through the data. In math:

```
price = w1*size + w2*bedrooms + w3*age + w4*distance + b
```

Where `w1, w2, w3, w4` are **weights** (coefficients) and `b` is the **bias** (intercept). The algorithm adjusts these weights to minimize prediction error.

In [ ]:
# Train the model — this is just ONE line!
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print('✅ Model trained!')
print()
print('What the model learned (coefficients):')
print(f'{"Feature":<15} {"Coefficient":>14}  {"Effect"}')
print('-' * 55)
for feat, coef in zip(FEATURES, model.coef_):
    effect = '↑ raises price' if coef > 0 else '↓ lowers price'
    print(f'{feat:<15} {coef:>+14,.0f}  {effect}')
print(f'\nBias (intercept): ${model.intercept_:,.0f}')

## Step 6 — Evaluate the Model

Three key metrics to know:

| Metric | Meaning | Good value |
|--------|---------|------------|
| **MAE** (Mean Absolute Error) | Average dollar amount the model is off by | Lower is better |
| **RMSE** (Root Mean Squared Error) | Similar to MAE, but penalizes big errors more | Lower is better |
| **R² Score** | How much of the price variation the model explains | Closer to 1.0 is better |

In [ ]:
# Make predictions on the test set
y_pred = model.predict(X_test_scaled)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print('📊 Model Performance:')
print(f'  Mean Absolute Error (MAE) : ${mae:>10,.0f}')
print(f'  Root Mean Squared Error   : ${rmse:>10,.0f}')
print(f'  R² Score                  : {r2:>12.4f}')
print()

# Interpret R²
if r2 >= 0.9:
    print('  🌟 Excellent! The model explains >90% of price variation.')
elif r2 >= 0.7:
    print('  ✅ Good. The model explains most of the price variation.')
else:
    print('  ⚠️  Moderate. Consider adding more features or trying a different model.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Model Performance', fontsize=14, fontweight='bold')

# --- Chart 1: Actual vs Predicted ---
ax = axes[0]
ax.scatter(y_test, y_pred, alpha=0.6, color='#2196F3', edgecolors='white', s=60)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect prediction')
ax.set_xlabel('Actual Price ($)')
ax.set_ylabel('Predicted Price ($)')
ax.set_title('Actual vs. Predicted')
ax.legend()
fmt = plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k')
ax.xaxis.set_major_formatter(fmt)
ax.yaxis.set_major_formatter(fmt)
ax.text(0.05, 0.95, f'R² = {r2:.4f}', transform=ax.transAxes,
        fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# --- Chart 2: Error distribution ---
ax = axes[1]
residuals = y_test.values - y_pred
ax.hist(residuals, bins=25, color='#4CAF50', edgecolor='white')
ax.axvline(0, color='red', linestyle='--', lw=2, label='Zero error')
ax.set_xlabel('Prediction Error ($)')
ax.set_ylabel('Frequency')
ax.set_title('Prediction Errors (Residuals)')
ax.legend()
ax.xaxis.set_major_formatter(fmt)

plt.tight_layout()
plt.savefig('house_price_results.png', dpi=120, bbox_inches='tight')
plt.show()
print()
print('💡 If the dots in Chart 1 are close to the red line → great predictions!')
print('   If Chart 2 is bell-shaped and centered on 0 → errors are random (good!)')

## Step 7 — Make Predictions on New Houses!

Now the fun part — let's predict prices for houses the model has never seen.

In [ ]:
# Define some new houses
new_houses = pd.DataFrame({
    'size_sqft':   [1200, 2500,  800, 3500],
    'bedrooms':    [   2,    4,    1,    5],
    'age_years':   [   5,   10,   30,    2],
    'distance_km': [ 3.0,  8.5, 20.0,  1.5],
})

# Scale and predict
new_houses_scaled = scaler.transform(new_houses)
predictions = model.predict(new_houses_scaled)

print('🏠 Price Predictions for New Houses:')
print()
print(f'  {"Size (sqft)":>12} {"Beds":>5} {"Age":>5} {"Dist":>7}  {"Predicted Price":>16}')
print('  ' + '-' * 55)
for (_, row), pred in zip(new_houses.iterrows(), predictions):
    print(f'  {row["size_sqft"]:>12.0f} {row["bedrooms"]:>5.0f} {row["age_years"]:>5.0f}'
          f' {row["distance_km"]:>5.1f}km  ${pred:>14,.0f}')

## Step 8 — Try Your Own House!

Edit the values below and run the cell to predict the price of your custom house.

In [ ]:
# ✏️ Edit these values!
my_house = pd.DataFrame({
    'size_sqft':   [1800],   # square feet
    'bedrooms':    [3],      # number of bedrooms
    'age_years':   [15],     # how old is the house
    'distance_km': [5.0],    # km from city center
})

scaled = scaler.transform(my_house)
price_pred = model.predict(scaled)[0]

print(f'🏡 Your House:')
for col in my_house.columns:
    print(f'   {col}: {my_house[col].values[0]}')
print()
print(f'💰 Estimated Price: ${price_pred:,.0f}')

## 🏁 Summary — What You Learned

1. **Data generation** — create or load a dataset with features and a target
2. **EDA** — explore data distributions and correlations before modeling
3. **Train/test split** — keep some data hidden to evaluate fairly
4. **Feature scaling** — normalize features to help the model
5. **Linear Regression** — fit a line through data to predict a continuous value
6. **Evaluation** — use MAE, RMSE, R² to measure how good your model is
7. **Prediction** — use the trained model on brand new data

---

**Next steps:**
- Try adding more features (garage, pool, school rating)
- Try a more powerful model: `RandomForestRegressor` or `GradientBoostingRegressor`
- Download a real dataset from [Kaggle House Prices](https://www.kaggle.com/c/house-prices-advanced-regression-techniques)
- Explore **cross-validation** for more robust evaluation